# GPU Portfolio & Risk Decision Engine — Colab GPU runner

A second, independent GPU data point for the benchmark (the primary one is a local RTX card; see
`docs/setup-wsl2.md`). It clones the repository at a pinned ref, installs the same pinned GPU stack,
and runs — **in this order, parity before speed** — the test suite, the CPU/GPU parity check, a
benchmark sweep and a backtest.

**Before Run all:** Runtime → Change runtime type → **T4 GPU** → Save.

> Colab's free tier has ~12.7 GB of system RAM, below cuOpt's stated 16 GB minimum. The sweep here
> stops at n=500 for that reason; if cuOpt still fails for memory, the cuDF/CuPy risk-model stages
> remain a valid second data point for how those stages scale across GPUs.

**When it finishes:** save a copy *with outputs* (File → Save a copy in GitHub/Drive). The outputs
are the evidence; the numbers themselves are committed from the downloaded results folder.

## 1. Confirm a GPU is attached

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "No NVIDIA GPU visible: Runtime -> Change runtime type -> T4 GPU, then Run all."

## 2. Clone the repository at a pinned ref

Pin `REF` to a release tag for reproducible numbers.

In [ ]:
REF = "main"  # e.g. "v1.0"
REPO = "https://github.com/jasonpereira518/gpu-portfolio-optimization-engine.git"
!rm -rf /content/gpu-portfolio-optimization-engine
!git clone --quiet {REPO} /content/gpu-portfolio-optimization-engine
%cd /content/gpu-portfolio-optimization-engine
!git checkout --quiet {REF} && git log --oneline -1

## 3. Install the pinned stack

cuDF, cuML and cuOpt come from one release (`requirements-gpu.txt`). The cu13 wheels need a driver
that supports CUDA 13; if this runtime's driver reports CUDA 12, the same pins are installed as cu12
wheels. This cell takes several minutes.

In [ ]:
import re
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
cuda_major = int(re.search(r"CUDA Version:\s*(\d+)", smi).group(1))
pins = open("requirements-gpu.txt").read()
if cuda_major < 13:
    pins = pins.replace("-cu13", "-cu12").replace("cuda13x", "cuda12x")
open("/tmp/requirements-gpu-local.txt", "w").write(pins)
print(f"driver supports CUDA {cuda_major}: installing", "cu13" if cuda_major >= 13 else "cu12", "wheels")
!pip install -q -r requirements.txt
!pip install -q --extra-index-url=https://pypi.nvidia.com -r /tmp/requirements-gpu-local.txt

In [ ]:
import importlib
for module in ["cudf", "cupy", "cuml", "cuopt"]:
    try:
        print(f"{module:6} {importlib.import_module(module).__version__}")
    except Exception as exc:
        print(f"{module:6} IMPORT FAILED: {type(exc).__name__}: {exc}")

## 4. Test suite

The GPU tests that skip on a CPU-only machine execute here.

In [ ]:
!python -m pytest tests/ -q -rs

## 5. Numerical parity — CPU vs GPU

Must pass **before** any timing is trusted: covariance to ~1e-9, objective value to 1e-8, plus
closed-form checks with no solver on the reference side.

In [ ]:
!python -m pipeline.parity_tests --n 200 --days 2000

## 6. Benchmark sweep — CPU vs GPU, per stage

In [ ]:
gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()[0]
OUT = "benchmarks/results/colab-" + re.sub(r"[^a-z0-9]+", "-", gpu_name.lower()).strip("-")
print("writing to", OUT)
!python -m benchmarks.run_benchmarks --sizes 50 500 --days 2520 --runs 5 --out {OUT}

## 7. Backtest — solution-quality parity across backends

In [ ]:
!python -m backtest.run_backtest --source synthetic --n 200 --frequency QE --out benchmarks/results/colab-backtest

## 8. Download the results

Commit the downloaded folder (with its `environment.json`) and run `make tables`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/colab_results", "zip", "benchmarks/results")
files.download("/content/colab_results.zip")